# Data Collection: Job Postings by Location and Key Skill

Part of my IBM Data Analyst Capstone project. In this notebook I collect job posting data from the course's Jobs API (a jobs dataset originally sourced from Naukri.com on Kaggle, public domain), count postings by **location** and by **key skill**, and save the results to Excel.

**Outputs:** `job-postings.xlsx` (by location) and `job-postings-skills.xlsx` (by skill).

In [2]:
%pip install openpyxl

In [3]:
# Import required libraries
import pandas as pd
import json
import requests

## 1. Job postings by technology
Load the full dataset from the API endpoint and count postings whose *Key Skills* field mentions a given technology.

In [4]:
api_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DA0321EN-SkillsNetwork/labs/module%201/Accessing%20Data%20Using%20APIs/jobs.json"

def get_number_of_jobs_T(technology):
    response = requests.get(api_url)
    data = response.json()

    number_of_jobs = 0
    for job in data:
        if job.get("Key Skills") and technology.lower() in job["Key Skills"].lower():
            number_of_jobs += 1

    return technology, number_of_jobs

In [5]:
get_number_of_jobs_T("Python")

('Python', 1173)

## 2. Job postings by location
Same approach, filtering on the *Location* field.

In [6]:
def get_number_of_jobs_L(location):
    response = requests.get(api_url)
    data = response.json()

    number_of_jobs = 0
    for job in data:
        if job.get("Location") and location.lower() in job["Location"].lower():
            number_of_jobs += 1

    return location, number_of_jobs

In [7]:
print(get_number_of_jobs_L("Los Angeles"))

('Los Angeles', 640)


## 3. Save postings by location to Excel
Run the location function for each target city and write the results to `job-postings.xlsx`.

In [8]:
from openpyxl import Workbook

In [9]:
locations = ["Los Angeles", "New York", "San Francisco", "Washington DC", "Seattle", "Austin", "Detroit"]

wb = Workbook()
ws = wb.active
ws.title = "Job Postings by Location"
ws.append(["Location", "Number of Job Postings"])

for loc in locations:
    location, count = get_number_of_jobs_L(loc)
    ws.append([location, count])
    print(location, count)

wb.save("job-postings.xlsx")

Los Angeles 640
New York 3226
San Francisco 435
Washington DC 5316
Seattle 3375
Austin 434
Detroit 3945


## 4. Job postings by key skill
Count postings for each programming language and database. Skills are matched as whole words so that, for example, "C" does not match "Scala" or "C++". Results are saved to `job-postings-skills.xlsx`.

In [10]:
import re
from openpyxl import Workbook

skills = ["C", "C#", "C++", "Java", "JavaScript", "Python", "Scala",
          "Oracle", "SQL Server", "MySQL", "PostgreSQL", "MongoDB"]

# Fetch the data once
data = requests.get(api_url).json()

def get_number_of_jobs_skill(skill):
    # match the skill as a whole word, so "C" doesn't match "Scala", "C#", "C++" etc.
    pattern = r'(?<![A-Za-z0-9+#])' + re.escape(skill.lower()) + r'(?![A-Za-z0-9+#])'
    count = 0
    for job in data:
        key_skills = job.get("Key Skills")
        if key_skills and re.search(pattern, key_skills.lower()):
            count += 1
    return skill, count

wb2 = Workbook()
ws2 = wb2.active
ws2.title = "Job Postings by Skill"
ws2.append(["Key Skill", "Number of Job Postings"])

for skill in skills:
    name, count = get_number_of_jobs_skill(skill)
    ws2.append([name, count])
    print(name, count)

wb2.save("job-postings-skills.xlsx")

C 961
C# 526
C++ 503
Java 1601
JavaScript 2246
Python 1171
Scala 89
Oracle 899
SQL Server 422
MySQL 951
PostgreSQL 86
MongoDB 208
